# Lab practice

In [25]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [26]:
train_df = pd.read_csv('data/sent_train.csv')
valid_df = pd.read_csv('data/sent_valid.csv')

In [27]:
if 'text' not in train_df.columns:
    print("No se encontró la columna 'text'. Se renombrará la primera columna como 'text'.")
    train_df.rename(columns={train_df.columns[0]: 'text'}, inplace=True)
    valid_df.rename(columns={valid_df.columns[0]: 'text'}, inplace=True)

sentiment_column = 'label'
print("Columnas luego de ajustes:", train_df.columns)

aresentiments = {
    "LABEL_0": "Bearish", 
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}


Columnas luego de ajustes: Index(['text', 'label'], dtype='object')


In [29]:
# Check if the sentiment column in train_df is of object type
if train_df[sentiment_column].dtype == object:
    # Map the sentiment labels to integers by splitting the string and taking the last part
    train_labels_mapped = train_df[sentiment_column].map(lambda x: int(x.split('_')[-1]))
    valid_labels_mapped = valid_df[sentiment_column].map(lambda x: int(x.split('_')[-1]))
    
    # If there are any unmapped values in train_df, warn and filter out those rows
    if train_labels_mapped.isnull().sum() > 0:
        print("Warning: Unmapped values found in train_df. Removing rows with invalid labels.")
        mask = train_labels_mapped.notnull()
        train_df = train_df[mask]
        train_labels_mapped = train_labels_mapped[mask]
    
    # Similarly, if there are unmapped values in valid_df, warn and filter them out
    if valid_labels_mapped.isnull().sum() > 0:
        print("Warning: Unmapped values found in valid_df. Removing rows with invalid labels.")
        mask = valid_labels_mapped.notnull()
        valid_df = valid_df[mask]
        valid_labels_mapped = valid_labels_mapped[mask]
else:
    # If the sentiment column is already numeric, use it directly
    train_labels_mapped = train_df[sentiment_column]
    valid_labels_mapped = valid_df[sentiment_column]

# Convert the mapped labels to integer numpy arrays
train_labels = train_labels_mapped.astype(int).values
valid_labels = valid_labels_mapped.astype(int).values

# Define the number of classes
num_classes = 3

# Convert the integer labels to one-hot encoded vectors
train_labels_cat = to_categorical(train_labels, num_classes=num_classes)
valid_labels_cat = to_categorical(valid_labels, num_classes=num_classes)

# Get the target names from the aresentiments dictionary
target_names = [aresentiments["LABEL_0"], aresentiments["LABEL_1"], aresentiments["LABEL_2"]]

In [30]:
vocab_size = 10000  
oov_token = "<OOV>"

# Inicializamos el tokenizador y lo entrenamos con los textos de entrenamiento
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(train_df['text'])

# Convertimos los textos a secuencias numéricas
train_sequences = tokenizer.texts_to_sequences(train_df['text'])
valid_sequences = tokenizer.texts_to_sequences(valid_df['text'])

# Definimos la longitud máxima de las secuencias y aplicamos padding
max_length = 50
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
valid_padded = pad_sequences(valid_sequences, maxlen=max_length, padding='post', truncating='post')
